In [2]:
# import libreries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

In [3]:
# create connection with MySQL server
load_dotenv()

user = os.getenv('MYSQL_USER')
password = os.getenv('MYSQL_PASSWORD')
host = os.getenv('DWH_HOST')
port = os.getenv('DWH_PORT')

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/retail_dwh")

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Connection OK' AS status, DATABASE() AS current_db, VERSION() AS mysql_version"))
    row = result.fetchone()
    print(f"Status:        {row[0]}")
    print(f"Database:      {row[1]}")
    print(f"MySQL version: {row[2]}")

Status:        Connection OK
Database:      retail_dwh
MySQL version: 8.0.46


In [4]:
# Show all the tables in the server
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))

    for row in result:
        print(row)

('dim_customer',)
('dim_date',)
('dim_geography',)
('dim_product',)
('fact_sales',)
('vw_daily_revenue',)
('vw_monthly_trend',)
('vw_return_rate',)
('vw_revenue_by_country',)
('vw_top_products',)


In [ ]:
# Function for queries
def consult(query):
    df = pd.read_sql(query, engine)
    return df

### Which products generate 80% of revenue?

In [6]:
pareto_query = """
WITH revenue_per_product AS (
    SELECT
        p.product_key,
        p.stock_code,
        p.description,
        SUM(f.total_revenue) AS revenue
    FROM fact_sales f
    JOIN dim_product p ON f.product_key = p.product_key
    WHERE f.is_return = 0 AND p.is_internal = 0
    GROUP BY p.product_key, p.stock_code, p.description
)
, revenue_with_pct AS (
    SELECT
        stock_code,
        description,
        revenue,
        SUM(revenue) OVER () AS total_revenue,
        revenue / SUM(revenue) OVER () * 100 AS revenue_pct
    FROM revenue_per_product
)
SELECT
    stock_code,
    description,
    revenue,
    total_revenue,
    revenue_pct,
    SUM(revenue_pct) OVER (ORDER BY revenue DESC) AS cumulative_pct
FROM revenue_with_pct
ORDER BY revenue DESC
LIMIT 50;
"""
pareto = consult(pareto_query)
print(f"Top 50 products revenue: £{pareto['revenue'].sum():,.0f}")
print(f"\nTop 10 products:")
print(pareto[['stock_code', 'description', 'revenue', 'cumulative_pct']].head(10))

Top 50 products revenue: £2,264,228

Top 10 products:
  stock_code                         description    revenue  cumulative_pct
0      22423            REGENCY CAKESTAND 3 TIER  174484.74        1.698625
1      23843         PAPER CRAFT , LITTLE BIRDIE  168469.60        3.338692
2     85123A  WHITE HANGING HEART T-LIGHT HOLDER  104518.80        4.356192
3      47566                       PARTY BUNTING   99504.33        5.324876
4     85099B             JUMBO BAG RED RETROSPOT   94340.05        6.243285
5      23166      MEDIUM CERAMIC TOP STORAGE JAR   81700.92        7.038651
6      23084                  RABBIT NIGHT LIGHT   66964.99        7.690561
7      22086     PAPER CHAIN KIT 50'S CHRISTMAS    64952.29        8.322877
8      84879       ASSORTED COLOUR BIRD ORNAMENT   59094.93        8.898171
9      79321                       CHILLI LIGHTS   54117.76        9.425012


In [7]:
# Find how many products make up 80% of revenue
query_full = """
SELECT 
    p.stock_code,
    p.description,
    SUM(f.total_revenue) AS revenue,
    SUM(SUM(f.total_revenue)) OVER (ORDER BY SUM(f.total_revenue) DESC) /
        SUM(SUM(f.total_revenue)) OVER () * 100 AS cumulative_pct
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
WHERE f.is_return = 0
  AND p.is_internal = 0
GROUP BY p.product_key, p.stock_code, p.description
ORDER BY revenue DESC;
"""

pareto_full = consult(query_full)

# Find 80% threshold
threshold_80 = pareto_full[pareto_full['cumulative_pct'] <= 80]
print(f"Total unique products: {len(pareto_full):,}")
print(f"Products generating 80% revenue: {len(threshold_80):,}")
print(f"That's {len(threshold_80)/len(pareto_full)*100:.1f}% of products generating 80% of revenue")
print(f"\nTotal revenue (excl. returns): £{pareto_full['revenue'].sum():,.0f}")

Total unique products: 3,912
Products generating 80% revenue: 824
That's 21.1% of products generating 80% of revenue

Total revenue (excl. returns): £10,272,119


### Finding 1 — Pareto Analysis (80/20 Rule)

The classic Pareto principle holds strongly for this retailer:
- **21.1% of products (824/3,912) generate 80% of total revenue (£8.2M)**
- Total annual revenue: £10.27M (excluding returns)
- The top product alone (REGENCY CAKESTAND 3 TIER) generates £174K — 1.7% of total revenue

**Business implication:** The retailer could focus inventory investment and 
marketing efforts on ~824 products and capture 80% of revenue. 
The remaining 3,088 products are candidates for catalogue review.

**Note:** PAPER CRAFT, LITTLE BIRDIE (rank #2, £168K) is flagged as anomalous —
a single order of 80,995 units that was immediately cancelled, 
suggesting a data entry error or failed wholesale deal.

In [8]:
query = """
SELECT 
    g.country,
    COUNT(DISTINCT f.customer_key)      AS unique_customers,
    COUNT(DISTINCT f.invoice_no)        AS total_orders,
    ROUND(SUM(f.total_revenue), 2)      AS total_revenue,
    ROUND(SUM(f.total_revenue) / 
        SUM(SUM(f.total_revenue)) OVER () * 100, 2) AS revenue_pct
FROM fact_sales f
JOIN dim_geography g ON f.geography_key = g.geography_key
WHERE f.is_return = 0
GROUP BY g.country
ORDER BY total_revenue DESC
LIMIT 15;
"""

geo = consult(query)
print(geo.to_string(index=False))

       country  unique_customers  total_orders  total_revenue  revenue_pct
United Kingdom              3921         18019     9025222.08        84.61
   Netherlands                 9            94      285446.34         2.68
          EIRE                 4           288      283453.96         2.66
       Germany                94           457      228867.14         2.15
        France                88           392      209715.11         1.97
     Australia                 9            57      138521.31         1.30
         Spain                30            90       61577.11         0.58
   Switzerland                22            54       57089.90         0.54
       Belgium                25            98       41196.34         0.39
        Sweden                 8            36       38378.33         0.36
         Japan                 8            19       37416.37         0.35
        Norway                10            36       36165.44         0.34
      Portugal           

### Finding 2 — Revenue by Country

The business is overwhelmingly UK-centric:
- **United Kingdom: £9.03M — 84.61% of total revenue**
- All 37 international markets combined: only 15.39% (£1.65M)
- Top international market: Netherlands £285K (2.68%)
- Notable: Australia ranks 6th despite only 9 customers — 
  highest revenue per customer internationally

**Business implication:** Significant growth opportunity in international markets.
Netherlands, EIRE, and Germany show strongest traction outside the UK.

In [9]:
query = """
SELECT 
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.invoice_no)    AS total_orders,
    ROUND(SUM(f.total_revenue), 2)  AS total_revenue
FROM fact_sales f
JOIN dim_date d ON f.date_key = d.date_key
WHERE f.is_return = 0
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month;
"""

monthly = consult(query)
print(monthly.to_string(index=False))

 year  month month_name  total_orders  total_revenue
 2010     12   December          1559      823746.14
 2011      1    January          1086      691364.56
 2011      2   February          1100      523631.89
 2011      3      March          1454      717639.36
 2011      4      April          1246      537808.62
 2011      5        May          1681      770536.02
 2011      6       June          1533      761739.90
 2011      7       July          1475      719221.19
 2011      8     August          1361      759138.38
 2011      9  September          1837     1058590.17
 2011     10    October          2040     1154979.30
 2011     11   November          2769     1509496.33
 2011     12   December           819      638792.68


In [10]:
query = """
SELECT 
    d.day_name,
    d.day_of_week,
    COUNT(DISTINCT f.invoice_no)    AS total_orders,
    ROUND(SUM(f.total_revenue), 2)  AS total_revenue,
    ROUND(AVG(f.total_revenue), 2)  AS avg_order_value
FROM fact_sales f
JOIN dim_date d ON f.date_key = d.date_key
WHERE f.is_return = 0
  AND d.is_weekend = 0
GROUP BY d.day_name, d.day_of_week
ORDER BY d.day_of_week;
"""

daily = consult(query)
print(daily.to_string(index=False))

 day_name  day_of_week  total_orders  total_revenue  avg_order_value
   Monday            1          3126     1779575.04            19.11
  Tuesday            2          3554     2178632.61            21.90
Wednesday            3          3690     1851147.81            20.05
 Thursday            4          4246     2203161.24            21.81
   Friday            5          3140     1840340.23            22.92


### Finding 3 — Seasonality

**Monthly pattern:**
- Peak month: November 2011 — £1.51M revenue (2,769 orders)
- Strong Q4 surge starting September (+39% MoM)
- Classic gift retail seasonality driven by Christmas purchasing

**Day of week pattern:**
- Thursday is the highest volume day (4,246 orders, £2.2M)
- Friday commands the highest average order value (£22.92)
- Zero weekend transactions — confirms predominantly B2B wholesale behavior

**Business implication:** Stock and staffing should be concentrated 
in Q4 (Sep–Nov). Marketing campaigns should target Thursday–Friday 
for maximum impact.

In [ ]:
query = """
SELECT 
    c.customer_id,
    c.segment,
    g.country,
    COUNT(DISTINCT f.invoice_no) AS total_orders,
    ROUND(SUM(f.total_revenue), 2) AS total_revenue,
    MAX(d.full_date) AS last_order_date,
    DATEDIFF('2011-12-09', MAX(d.full_date)) AS days_since_last_order
FROM fact_sales f
JOIN dim_customer c ON f.customer_key  = c.customer_key
JOIN dim_geography g ON f.geography_key = g.geography_key
JOIN dim_date d ON f.date_key = d.date_key
WHERE f.is_return = 0
  AND c.customer_id != 'UNKNOWN'
GROUP BY c.customer_key, c.customer_id, c.segment, g.country
HAVING days_since_last_order >= 60
   AND total_revenue > 1000
ORDER BY total_revenue DESC
LIMIT 20;
"""

churn_risk = consult(query)
print(churn_risk.to_string(index=False))

customer_id segment        country  total_orders  total_revenue last_order_date  days_since_last_order
      12346     B2B United Kingdom             1       77183.60      2011-01-18                    325
      15749     B2B United Kingdom             3       44534.30      2011-04-18                    235
      15098     B2C United Kingdom             3       39916.50      2011-06-10                    182
      12939     B2B United Kingdom             8       11581.80      2011-10-06                     64
      12409     B2C    Switzerland             3       11072.67      2011-09-22                     78
      16180     B2C United Kingdom             8       10254.18      2011-08-31                    100
      12590     B2C        Germany             2        9864.26      2011-05-12                    211
      13093     B2C United Kingdom             8        7832.47      2011-03-09                    275
      12435     B2C        Denmark             2        7829.89      2011

### Finding 4 — Customer Churn Risk

20 high-value customers (>£1,000 revenue) haven't ordered in 60+ days.

**Top 3 at-risk customers:**
| Customer | Revenue | Last Order | Days Silent | Priority |
|---|---|---|---|---|
| 12346 | £77,184 | Jan 2011 | 325 days | CRITICAL |
| 15749 | £44,534 | Apr 2011 | 235 days | HIGH |
| 15098 | £39,917 | Jun 2011 | 182 days | HIGH |

**Notable:** Customer 17850 placed 34 orders but went silent after 
Dec 2010 — earliest and most loyal customer lost.

**Business implication:** Immediate outreach to customers 12346 and 
15749 could recover £120K+ in at-risk revenue.

In [12]:
query = """
SELECT 
    p.stock_code,
    p.description,
    SUM(CASE WHEN f.is_return = 0 THEN f.quantity ELSE 0 END)  AS units_sold,
    SUM(CASE WHEN f.is_return = 1 THEN ABS(f.quantity) ELSE 0 END) AS units_returned,
    ROUND(
        SUM(CASE WHEN f.is_return = 1 THEN ABS(f.quantity) ELSE 0 END) * 100.0 /
        NULLIF(SUM(CASE WHEN f.is_return = 0 THEN f.quantity ELSE 0 END), 0)
    , 2) AS return_rate_pct,
    ROUND(SUM(CASE WHEN f.is_return = 0 THEN f.total_revenue ELSE 0 END), 2) AS gross_revenue
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
WHERE p.is_internal = 0
GROUP BY p.product_key, p.stock_code, p.description
HAVING units_sold >= 100
   AND return_rate_pct > 0
ORDER BY return_rate_pct DESC
LIMIT 15;
"""

returns = consult(query)
print(returns.to_string(index=False))

stock_code                         description  units_sold  units_returned  return_rate_pct  gross_revenue
     23843         PAPER CRAFT , LITTLE BIRDIE     80995.0         80995.0           100.00      168469.60
     84347 ROTATING SILVER ANGELS T-LIGHT HLDR      9476.0          9376.0            98.94       26463.83
     23166      MEDIUM CERAMIC TOP STORAGE JAR     78033.0         74494.0            95.46       81700.92
    85232B     SET OF 3 BABUSHKA STACKING TINS       275.0           255.0            92.73        1361.25
     23113               PANTRY CHOPPING BOARD      1154.0           946.0            81.98        5868.50
    85023B     EAU DE NILE JEWELLED PHOTOFRAME       105.0            73.0            69.52         235.35
     23055     IVORY CHANDELIER T-LIGHT HOLDER       658.0           432.0            65.65        1401.43
     23056   FLOWERS CHANDELIER T-LIGHT HOLDER       966.0           579.0            59.94        2916.87
    47566B              TEA TIME PART

### Finding 5 — Product Return Rate Analysis

**Critical quality issues identified:**

| Product | Return Rate | Units Returned | Implication |
|---|---|---|---|
| Rotating Silver Angels T-Light | 98.94% | 9,376 | Discontinue immediately |
| Medium Ceramic Storage Jar | 95.46% | 74,494 | Major quality defect |
| Pantry Chopping Board | 81.98% | 946 | Quality review needed |
| Chandelier T-Light range (3 SKUs) | 55–65% | 1,444 | Product line review |

**Business implication:** The Chandelier T-Light product line 
(23055, 23056, 23057) shows a systematic return pattern suggesting 
a design or quality issue affecting the entire range — not isolated incidents.
Fixing this line alone could recover ~£6K in net revenue.

In [13]:
query = """
WITH rfm_base AS (
    SELECT
        c.customer_id,
        c.segment,
        DATEDIFF('2011-12-09', MAX(d.full_date)) AS recency_days,
        COUNT(DISTINCT f.invoice_no) AS frequency,
        ROUND(SUM(f.total_revenue), 2) AS monetary
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    JOIN dim_date d ON f.date_key = d.date_key
    WHERE f.is_return = 0
      AND c.customer_id != 'UNKNOWN'
    GROUP BY c.customer_key, c.customer_id, c.segment
),
rfm_scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency_days ASC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency DESC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary DESC) AS m_score
    FROM rfm_base
)
SELECT
    r_score, f_score, m_score,
    ROUND(AVG(recency_days), 0) AS avg_recency,
    ROUND(AVG(frequency), 1) AS avg_frequency,
    ROUND(AVG(monetary), 2) AS avg_monetary,
    COUNT(*) AS customer_count,
    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Loyal'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customer'
        WHEN r_score <= 2 AND f_score >= 3 AND m_score >= 3 THEN 'At Risk'
        WHEN r_score <= 2 AND f_score <= 2 AND m_score <= 2 THEN 'Lost'
        ELSE 'Potential'
    END AS customer_segment
FROM rfm_scored
GROUP BY r_score, f_score, m_score
ORDER BY r_score DESC, f_score DESC, m_score DESC;
"""

rfm = consult(query)

# Summarize by segment
summary = rfm.groupby('customer_segment').agg(
    customers   =('customer_count', 'sum'),
    avg_monetary=('avg_monetary', 'mean')
).round(2).sort_values('customers', ascending=False)

print(summary)

                  customers  avg_monetary
customer_segment                         
Lost                   1008       3760.09
Champions               918        253.37
Loyal                   883        459.63
Potential               864       3736.15
At Risk                 449        393.47
New Customer            216       1502.22


### Executive Summary — UK Online Gift Retailer 2010-2011

### Business Performance
- Total revenue: £10.27M across 539,392 transactions
- 38 countries served, 4,371 unique customers

### 5 Key Findings

1. **Pareto** — 21.1% of products (824) drive 80% of revenue
2. **Geography** — UK dominates at 84.6%; Netherlands and EIRE 
   are strongest international markets
3. **Seasonality** — November peak (£1.51M); Thursday is highest 
   volume day; zero weekend activity confirms B2B nature
4. **Churn** — 20 high-value customers at risk; top 3 represent 
   £161K in recoverable revenue
5. **Quality** — Rotating Silver Angels (98.9% return) and 
   Ceramic Storage Jar (95.5% return) require immediate review

### Strategic Recommendations
1. Launch win-back campaign for 1,008 lost customers (avg £3,760 value)
2. Convert 864 Potential customers (avg £3,736) to Loyal tier
3. Discontinue or redesign Chandelier T-Light product line
4. Invest in international expansion — Netherlands and EIRE show 
   strong traction with minimal customer base